In [1]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import sys
sys.path.append("..")
from shared_util.time_handler import TimeHandler

In [ ]:
# 与源目录结构保持一致，争对aiops代码
'''
datasets/
└── train/
    └── 2022-03-21/
        └── cloudbed-1/
            └── trace/
                └── all/
                    └── trace_jaeger-span.csv
'''

In [2]:
# =========================
# 0) 工具：mkdir
# =========================
def ensure_dir(p: str):
    os.makedirs(p, exist_ok=True)
    return p

# =========================
# 1) 路径解析：dataset_type/date/cloudbed
# =========================
def parse_aiops_ids_from_trace_path(trace_path: str, data_base: str):
    """
    trace_path:
      {data_base}/{dataset_type}/{date}/{cloudbed}/trace/all/trace_jaeger-span.csv
    """
    rel = os.path.relpath(trace_path, data_base)
    parts = rel.split(os.sep)

    # [dataset_type, date, cloudbed, 'trace', 'all', 'trace_jaeger-span.csv']
    if len(parts) < 6:
        raise ValueError(f"Unexpected path structure: {rel}")

    return parts[0], parts[1], parts[2]

# =========================
# 2) 时间桶：24小时，每分钟（秒级）
# =========================
def build_day_minute_timestamps(date: str, TimeHandler):
    start_ts = TimeHandler.datetime_to_timestamp(f"{date} 00:00:00")  # 秒
    return [start_ts + 60 * i for i in range(24 * 60)]

# =========================
# 3) 核心：复刻原 process_traces
# =========================
def process_traces_aiops(
    timestamp_list: list,          # 秒级，1min间隔
    trace_csv_path: str,
    out_dir: str,
    pod_list: list,                # pod_order
    rename_service2pod,            # service -> pods
    service_order_list: list,      # service_order
    chunksize: int = 100_000,
):
    """
    输出：out_dir/span_features.csv
    列结构完全复刻原实现：
      <intensity>; cmdb_id: POD; type: upstream/current/downstream
      <duration>;  cmdb_id: POD; type: upstream/current/downstream
    """
    ensure_dir(out_dir)

    # 1) trace_pattern_list = [cmdb_id: {pod}]
    trace_pattern_list = []
    for service in service_order_list:
        for pod in rename_service2pod(service):
            trace_pattern_list.append(f"cmdb_id: {pod}")

    
    T = len(timestamp_list)
    '''
        temp_bucket_dict[cmdb_id: {pod}][i] = {"parent_span": [], "duration": [], "span_index_dict": {}}
        span_index_dict[{trace_id}/{span_id}] = 某一时刻，该pod的被调用数量
        parent_span 是所有调用该pod的{trace_id}/{span_id}
        duration 是所有调用该pod的duration
    '''
    temp_bucket_dict = {
        tp: [
            {"parent_span": [], "duration": [], "span_index_dict": {}}
            for _ in range(T)
        ]
        for tp in trace_pattern_list
    }
   

    # 3) id_to_trace_pattern[i]['{trace_id}/{span_id}'] = 'cmdb_id: {cmdb_id}'
    id_to_trace_pattern = [dict() for _ in range(T)]

    start_ts = timestamp_list[0]  # 秒

    reader = pd.read_csv(trace_csv_path, chunksize=chunksize)
    '''
        timestamp: ms                   
        cmdb_id: pod           
        span_id
        trace_id
        duration
        type
        status_code
        operation_name
        parent_span  
    '''
    # 统计每个 pod 在每一分钟内的 span 信息
    for chunk in reader:
        for row in chunk.itertuples(index=False):
            cmdb_id = getattr(row, "cmdb_id", None)
            ts_ms   = getattr(row, "timestamp", None)   # 原实现用 row["timestamp"]/1000
            trace_id = getattr(row, "trace_id", None)
            span_id  = getattr(row, "span_id", None)
            parent_span = getattr(row, "parent_span", None)
            duration = getattr(row, "duration", None)

            duration = float(duration)
            duration = np.log1p(duration)
            if cmdb_id is None or ts_ms is None or trace_id is None or span_id is None:
                continue
            if cmdb_id not in pod_list:
                continue

            # 原逻辑：index = int((timestamp/1000 - start)/60)
            ts_s = float(ts_ms) / 1000.0
            idx = int((ts_s - start_ts) / 60)

            if idx < 0 or idx >= T:
                continue

            trace_pattern = f"cmdb_id: {cmdb_id}"
            span_key = f"{trace_id}/{span_id}"

            # 在该分钟内，这个 span 属于哪个 pod
            id_to_trace_pattern[idx][span_key] = trace_pattern

            bucket = temp_bucket_dict[trace_pattern][idx]
            # 记录每个 span 在 duration 列表中的位置
            bucket["span_index_dict"][span_key] = len(bucket["duration"])
            bucket["parent_span"].append(f"{trace_id}/{parent_span}")
            bucket["duration"].append(duration)

    # 4) 先算 current 特征
    span_feature_dict = {"timestamp": timestamp_list}
    '''
        当前pod自己和作为被调用pod和调用pod的密度（span数）和调用平均时延
        最后作为一张timestamp 六个元素组成的横表。
    '''

    for tp in trace_pattern_list:
        '''
            timestamp
            <intensity>; cmdb_id: adservice-0; type: current
            <duration>;  cmdb_id: adservice-0; type: current
            <intensity>; cmdb_id: adservice-0; type: upstream
        '''
        span_feature_dict[f"<intensity>; {tp}; type: upstream"] = []
        span_feature_dict[f"<duration>; {tp}; type: upstream"] = []
        span_feature_dict[f"<intensity>; {tp}; type: current"] = []
        span_feature_dict[f"<duration>; {tp}; type: current"] = []
        span_feature_dict[f"<intensity>; {tp}; type: downstream"] = []
        span_feature_dict[f"<duration>; {tp}; type: downstream"] = []

        for i in range(T):
            cur_durs = temp_bucket_dict[tp][i]["duration"]
            # 该分钟 span 数量
            span_feature_dict[f"<intensity>; {tp}; type: current"].append(len(cur_durs))
            # 该分钟 span 平均时延
            if len(cur_durs) > 0:
                span_feature_dict[f"<duration>; {tp}; type: current"].append(
                    float(np.nan_to_num(np.nanmean(cur_durs)))
                )
            else:
                span_feature_dict[f"<duration>; {tp}; type: current"].append(0.0)

    # 5) 再算 upstream/downstream（逐桶）
    # parent pod 在这一分钟里产生了多少个 “下游调用”（child spans）
    # 并用 child span 的 duration 表示这些下游调用的耗时
    for i in range(T):
        child_span_dict = {"upstream": {}, "downstream": {}}

        for tp in trace_pattern_list:
            parents = temp_bucket_dict[tp][i]["parent_span"]
            durs    = temp_bucket_dict[tp][i]["duration"]
            span_index_dict = temp_bucket_dict[tp][i]["span_index_dict"]

            for k in range(len(parents)):
                parent_span_id = parents[k]

                # parent 不在同桶则跳
                if parent_span_id not in id_to_trace_pattern[i]:
                    continue

                parent_tp = id_to_trace_pattern[i][parent_span_id]

                # ---- downstream（以 parent_tp 为 key；duration 用 child duration）
                if parent_tp not in child_span_dict["downstream"]:
                    child_span_dict["downstream"][parent_tp] = {"intensity": 0, "duration": []}
                child_span_dict["downstream"][parent_tp]["intensity"] += 1
                child_span_dict["downstream"][parent_tp]["duration"].append(durs[k])

                # ---- upstream（以 child tp 为 key；duration 用 parent duration）
                # parent_tp 桶里必须存在 parent_span_id 才能拿到 parent duration
                if parent_span_id not in temp_bucket_dict[parent_tp][i]["span_index_dict"]:
                    continue

                parent_index = temp_bucket_dict[parent_tp][i]["span_index_dict"][parent_span_id]
                parent_dur = temp_bucket_dict[parent_tp][i]["duration"][parent_index]

                if tp not in child_span_dict["upstream"]:
                    child_span_dict["upstream"][tp] = {"intensity": 0, "duration": []}
                child_span_dict["upstream"][tp]["intensity"] += 1
                child_span_dict["upstream"][tp]["duration"].append(parent_dur)

        # 写入每个 tp 的 upstream/downstream；无则补 0
        # child pod 在这一分钟里被多少个上游 span 触发（也就是 parent spans 数）
        # 并用 parent span 的 duration 表示上游侧的耗时
        for tp in trace_pattern_list:
            for ft in ["upstream", "downstream"]:
                col_i = f"<intensity>; {tp}; type: {ft}"
                col_d = f"<duration>; {tp}; type: {ft}"

                if tp in child_span_dict[ft]:
                    v = child_span_dict[ft][tp]
                    span_feature_dict[col_i].append(int(v["intensity"]))
                    if len(v["duration"]) > 0:
                        span_feature_dict[col_d].append(float(np.nan_to_num(np.nanmean(v["duration"]))))
                    else:
                        span_feature_dict[col_d].append(0.0)
                else:
                    span_feature_dict[col_i].append(0)
                    span_feature_dict[col_d].append(0.0)

    # 6) 输出
    out_path = os.path.join(out_dir, "span_features.csv")
    '''
        行 = 时间（每分钟）列 = 所有 pod × 6 个 trace 特征
        <intensity>; cmdb_id: POD; type: upstream
        <duration>;  cmdb_id: POD; type: upstream
        <intensity>; cmdb_id: POD; type: current
        <duration>;  cmdb_id: POD; type: current
        <intensity>; cmdb_id: POD; type: downstream
        <duration>;  cmdb_id: POD; type: downstream
            
    '''
    pd.DataFrame(span_feature_dict).to_csv(out_path, index=False)
    return out_path

# =========================
# 4) 总控：extract_trace_features（os.walk + 镜像输出）
# =========================
def extract_trace_features_aiops(
    data_base: str,
    out_base: str,
    TimeHandler,
    pod_list: list,
    service_order_list: list,
    rename_service2pod,
    target_name="trace_jaeger-span.csv",
    skip_if_path_contains=("test_data",),
    skip_bad_case=True,
    chunksize: int = 100_000,
):
    for root, _, files in os.walk(data_base):
        if target_name not in files:
            continue

        trace_path = os.path.join(root, target_name)

        # # 跳过 test_data
        # if any(x in trace_path for x in skip_if_path_contains):
        #     continue

        dataset_type, date, cloudbed = parse_aiops_ids_from_trace_path(trace_path, data_base)

        # 原逻辑 bad case
        if skip_bad_case and date == "2022-03-24" and cloudbed in ["cloudbed-1", "cloudbed-2"]:
            continue

        timestamp_list = build_day_minute_timestamps(date, TimeHandler)

        out_dir = ensure_dir(os.path.join(out_base, dataset_type, date, cloudbed, "raw_trace"))

        print(f"[INFO] trace: {dataset_type} {date} {cloudbed} -> {out_dir}")

        process_traces_aiops(
            timestamp_list=timestamp_list,
            trace_csv_path=trace_path,
            out_dir=out_dir,
            pod_list=pod_list,
            rename_service2pod=rename_service2pod,
            service_order_list=service_order_list,
            chunksize=chunksize,
        )

        print(f"[OK] saved span_features.csv in {out_dir}")

In [3]:
pod_list = [
    'adservice-0', 
    'adservice-1', 
    'adservice-2', 
    'adservice2-0', 
    'cartservice-0', 
    'cartservice-1', 
    'cartservice-2', 
    'cartservice2-0', 
    'checkoutservice-0', 
    'checkoutservice-1', 
    'checkoutservice-2', 
    'checkoutservice2-0', 
    'currencyservice-0', 
    'currencyservice-1', 
    'currencyservice-2', 
    'currencyservice2-0', 
    'emailservice-0', 
    'emailservice-1', 
    'emailservice-2', 
    'emailservice2-0', 
    'frontend-0', 
    'frontend-1', 
    'frontend-2', 
    'frontend2-0', 
    'paymentservice-0', 
    'paymentservice-1', 
    'paymentservice-2', 
    'paymentservice2-0', 
    'productcatalogservice-0', 
    'productcatalogservice-1', 
    'productcatalogservice-2', 
    'productcatalogservice2-0', 
    'recommendationservice-0', 
    'recommendationservice-1', 
    'recommendationservice-2', 
    'recommendationservice2-0', 
    'shippingservice-0', 
    'shippingservice-1', 
    'shippingservice-2', 
    'shippingservice2-0'
]

In [4]:
service_order_list = [
    "adservice","cartservice","checkoutservice","currencyservice","emailservice",
    "frontend","paymentservice","productcatalogservice","recommendationservice","shippingservice"
]

In [5]:
def rename_service2pod(service: str):
    # 仅返回 -1/-2（与 pod_list 一致）
    return [f"{service}-0", f"{service}-1", f"{service}-2", f"{service}2-0"]

In [ ]:
extract_trace_features_aiops(
    data_base="/home/ZZData/Eadro/preprocess/datasets/aiops2022/",
    out_base="../../temp_data/aiops22/raw_data",
    TimeHandler=TimeHandler,
    pod_list=pod_list,
    service_order_list=service_order_list,
    rename_service2pod=rename_service2pod,
    skip_if_path_contains=("test_data",),
    skip_bad_case=True,
)

In [8]:
import pandas as pd
import numpy as np
import glob

# 随便挑一个 case
f = glob.glob("../../temp_data/aiops22/raw_data/training_data_with_faults/2022-03-20/cloudbed-1/raw_trace/span_features.csv", recursive=True)[0]
df = pd.read_csv(f)

dur_cols = [c for c in df.columns if "<duration>" in c]
int_cols = [c for c in df.columns if "<intensity>" in c]

print("FILE:", f)
print("duration max:", np.nanmax(df[dur_cols].values))
print("intensity max:", np.nanmax(df[int_cols].values))
print("duration 99p:", np.nanpercentile(df[dur_cols].values, 99))
print("intensity 99p:", np.nanpercentile(df[int_cols].values, 99))

FILE: ../../temp_data/aiops22/raw_data/training_data_with_faults/2022-03-20/cloudbed-1/raw_trace/span_features.csv
duration max: 4.110921630271143
intensity max: 2878
duration 99p: 0.068632192784879
intensity 99p: 1512.0099999999948
